In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

device

'cuda'

In [3]:
SEED = 42

In [4]:
from src.backend import core_factory


core = core_factory(device)

In [5]:
import pandas as pd
import numpy as np


def create_features(df: pd.DataFrame):
    X = df.copy()

    X["study_efficiency"] = X["study_hours"] * X["class_attendance"] / 100
    X["sleep_study_ratio"] = X["sleep_hours"] / (X["study_hours"] + 0.001)

    X["study_intensity"] = pd.cut(
        X["study_hours"],
        bins=[-np.inf, 3, 6, np.inf],
        labels=["low", "medium", "high"],
        include_lowest=True,
    )

    X["attendance_level"] = pd.cut(
        X["class_attendance"],
        bins=[-np.inf, 50, 75, 90, np.inf],
        labels=["poor", "average", "good", "excellent"],
        include_lowest=True,
    )

    X["internet_access"] = X["internet_access"].map({"no": 0, "yes": 1})
    X["sleep_quality"] = X["sleep_quality"].map({"poor": 0, "average": 1, "good": 2})
    X["facility_rating"] = X["facility_rating"].map({"low": 0, "medium": 1, "high": 2})
    X["exam_difficulty"] = X["exam_difficulty"].map(
        {"easy": 0, "moderate": 1, "hard": 2}
    )
    X["study_intensity"] = (
        X["study_intensity"].map({"low": 0, "medium": 1, "high": 2}).astype(int)
    )
    X["attendance_level"] = (
        X["attendance_level"]
        .map({"poor": 0, "average": 1, "good": 2, "excellent": 3})
        .astype(int)
    )

    course_ohe = pd.get_dummies(X["course"], prefix="course", dummy_na=True)
    study_method_ohe = pd.get_dummies(
        X["study_method"], prefix="study_method", dummy_na=True
    )
    gender_ohe = pd.get_dummies(
        X["gender"],
        prefix="gender",
        dummy_na=True,
    )

    X.drop("course", axis=1, inplace=True)
    X.drop("study_method", axis=1, inplace=True)
    X.drop("gender", axis=1, inplace=True)

    X = pd.concat([X, course_ohe, study_method_ohe, gender_ohe], axis=1)

    return X

In [6]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")

train.drop("id", axis=1, inplace=True)

In [7]:
train_fe = create_features(train)
test_fe = create_features(test)

In [8]:
y_train = train_fe["exam_score"]
X_train: pd.DataFrame = train_fe.drop("exam_score", axis=1)
X_test: pd.DataFrame = test_fe

In [9]:
numerical_features = [
    "age",
    "study_hours",
    "class_attendance",
    "sleep_hours",
    "study_efficiency",
    "sleep_study_ratio",
]

In [10]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

In [11]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from src.lgbm_cuda import get_lgbm_device


def lgbm_fit(
    model: LGBMRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    model.fit(
        X_train,
        y_train,
        eval_X=X_valid,
        eval_y=y_valid,
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                verbose=False,
            ),
            lgb.log_evaluation(period=0),
        ],
    )

    return model


def lgbm_predict(model: LGBMRegressor, X_valid: pd.DataFrame):
    return model.predict(
        X_valid,
        num_iteration=model.best_iteration_,
    )

In [12]:
from xgboost import XGBRegressor
import cupy as cp
import numpy as np


def xgb_fit(
    model: XGBRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    model.set_params(early_stopping_rounds=50, callbacks=None)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )

    return model


def xgb_predict(model: XGBRegressor, X_valid: pd.DataFrame):
    if model.device == "cuda":
        if hasattr(X_valid, "to_numpy"):
            X_valid = X_valid.to_numpy(dtype=np.float32)

        X_valid = cp.asarray(X_valid)

    return model.predict(
        X_valid,
        iteration_range=(0, model.best_iteration + 1),
    )

In [13]:
from catboost import CatBoostRegressor


def catboost_fit(
    model: CatBoostRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    model.set_params(early_stopping_rounds=50)

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        verbose=False,
    )

    return model


def catboost_predict(model: CatBoostRegressor, X_valid: pd.DataFrame):
    return model.predict(X_valid)

In [14]:
from sklearn.linear_model import Ridge


def ridge_fit(
    model: Ridge,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    return model.fit(X_train, y_train)


def ridge_predict(model: Ridge, X_valid: pd.DataFrame):
    return model.predict(X_valid)

In [15]:
from sklearn.ensemble import RandomForestRegressor


def forest_fit(
    model: RandomForestRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    return model.fit(X_train, y_train)


def forest_predict(model: RandomForestRegressor, X_valid: pd.DataFrame):
    return model.predict(X_valid)

In [16]:
from src.nn.regressor import LightningTabularRegressor
from src.nn.litmlp import LitMLP
from src.nn.litresnet import LitTabResNet
from src.nn.littabm import LitTabM


def lit_fit(
    model: LightningTabularRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    return model.fit(X_train, y_train, X_valid, y_valid)


def lit_predict(
    model: LightningTabularRegressor,
    X_valid: pd.DataFrame,
):
    return model.predict(X_valid)

In [17]:
from src.config_parser import ModelConfig
from src.config import *

SCHEMAS = {
    "Ridge": RidgeConfig,
    "Lasso": LassoConfig,
    "ElasticNet": ElasticNetConfig,
    "KNN": KNNConfig,
    "DecisionTree": DecisionTreeConfig,
    "RandomForest": RandomForestConfig,
    "XGBoost": XGBoostConfig,
    "CatBoost": CatBoostConfig,
    "LightGBM": LightGBMConfig,
    "TabMLP": TabMLPConfig,
    "TabResNet": TabResNetConfig,
    "TabM": TabMConfig,
}

config = ModelConfig("../config.ini", SCHEMAS)

In [18]:
from src.predict import ensemble_predict

train, oof_pred_df, test_pred_df = ensemble_predict(
    {
        "LightGBM": (
            LGBMRegressor(
                **config["LightGBM"],
                objective="regression",
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
                device=get_lgbm_device(device),
            ),
            lgbm_fit,
            lgbm_predict,
            False,
            True,
        ),
        "XGBoost": (
            XGBRegressor(
                **config["XGBoost"],
                random_state=SEED,
                objective="reg:squarederror",
                n_jobs=-1,
                eval_metric="rmse",
                tree_method="hist",
                device=device,
            ),
            xgb_fit,
            xgb_predict,
            False,
            True,
        ),
        "CatBoost": (
            CatBoostRegressor(
                **config["CatBoost"],
                loss_function="RMSE",
                eval_metric="RMSE",
                random_seed=SEED,
                thread_count=-1,
                verbose=False,
                allow_writing_files=False,
                task_type="GPU" if device == "cuda" else "CPU",
            ),
            catboost_fit,
            catboost_predict,
            False,
            True,
        ),
        "Ridge": (
            core.linear_model.Ridge(**config["Ridge"]),
            ridge_fit,
            ridge_predict,
            True,
            True,
        ),
        "RandomForest": (
            core.ensemble.RandomForestRegressor(
                **config["RandomForest"],
                random_state=SEED,
            ),
            forest_fit,
            forest_predict,
            False,
            True,
        ),
        "TabMLP": (
            LightningTabularRegressor(
                model=LitMLP,
                **config["TabMLP"],
                num_workers=0,
                seed=SEED,
                numeric_cols=numerical_features,
                device=device,
                verbose=False,
            ),
            lit_fit,
            lit_predict,
            True,
            False,
        ),
        "TabResNet": (
            LightningTabularRegressor(
                model=LitTabResNet,
                **config["TabResNet"],
                num_workers=0,
                seed=SEED,
                numeric_cols=numerical_features,
                device=device,
                verbose=False,
            ),
            lit_fit,
            lit_predict,
            True,
            False,
        ),
        "TabM": (
            LightningTabularRegressor(
                model=LitTabM,
                **config["TabM"],
                num_workers=0,
                seed=SEED,
                numeric_cols=numerical_features,
                device=device,
                verbose=False,
            ),
            lit_fit,
            lit_predict,
            True,
            False,
        ),
    },
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    kf=kf,
    numerical_features=numerical_features,
    checkpoint_dir="../hillclimbers",
    recompute=False,
    ignore=[],
)

[checkpoint] LightGBM: loaded
[checkpoint] XGBoost: loaded
[checkpoint] CatBoost: loaded
[checkpoint] Ridge: loaded
[checkpoint] RandomForest: loaded
[checkpoint] TabMLP: loaded
[checkpoint] TabResNet: loaded
[checkpoint] TabM: loaded


In [19]:
from hillclimbers import climb_hill, partial
from sklearn.metrics import root_mean_squared_error

result = climb_hill(
    train=train,
    oof_pred_df=oof_pred_df,
    test_pred_df=test_pred_df,
    target="target",
    objective="minimize",
    eval_metric=partial(root_mean_squared_error),
    negative_weights=True,
    precision=0.01,
    plot_hill=True,
    plot_hist=True,
)

submission = X_test[["id"]]
submission["exam_score"] = result
submission.to_csv("../submission.csv", index=False)

   /\  
  /__\  hillclimbers 
 /    \
/______\ 

Models to be ensembled | (8 total): 

XGBoost:      8.73149 (best solo model)
LightGBM:     8.74163
CatBoost:     8.75925
TabM:         8.85450
TabMLP:       8.85672
RandomForest: 8.88333
TabResNet:    8.88662
Ridge:        8.88773

[Data preparation completed successfully] - [Initiate hill climbing] 

Iteration: 1 | Model added: RandomForest | Best weight: -0.17 | Best root_mean_squared_error: 8.72812
Iteration: 2 | Model added: LightGBM | Best weight: 0.26 | Best root_mean_squared_error: 8.72624
Iteration: 3 | Model added: TabM | Best weight: -0.14 | Best root_mean_squared_error: 8.72436
Iteration: 4 | Model added: CatBoost | Best weight: 0.13 | Best root_mean_squared_error: 8.72354
Iteration: 5 | Model added: Ridge | Best weight: -0.05 | Best root_mean_squared_error: 8.72312
Iteration: 6 | Model added: TabResNet | Best weight: 0.01 | Best root_mean_squared_error: 8.72312
Iteration: 7 | Model added: TabMLP | Best weight: -0.01 | Best r